In [1]:
!pip install jpegio

import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import jpegio as jio
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import requests, tarfile

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

# ── Data Download ─────────────────────────────────────────────
TAR_URL     = "https://huggingface.co/datasets/plok5308/djpeg_dataset/resolve/main/jpeg_data.tar.gz"
EXTRACT_DIR = '/kaggle/working/extracted'
os.makedirs(os.path.join(EXTRACT_DIR, 'single'), exist_ok=True)
os.makedirs(os.path.join(EXTRACT_DIR, 'double'), exist_ok=True)

def stream_extract(url, extract_dir, limit=40000):
    s_count, d_count = 0, 0
    print("Streaming extraction started...")
    response = requests.get(url, stream=True)
    with tarfile.open(fileobj=response.raw, mode="r|gz") as tar:
        for member in tqdm(tar, desc="Processing"):
            if member.isfile() and member.name.lower().endswith('.jpg'):
                label = None
                if 'single' in member.name.lower() and s_count < limit:
                    label = 'single'; s_count += 1
                elif 'double' in member.name.lower() and d_count < limit:
                    label = 'double'; d_count += 1
                if label:
                    member.name = os.path.basename(member.name)
                    tar.extract(member, path=os.path.join(extract_dir, label), filter='data')
            if s_count >= limit and d_count >= limit:
                break
    print(f"Extraction Done: Single={s_count}, Double={d_count}")

stream_extract(TAR_URL, EXTRACT_DIR)


# ── Model ─────────────────────────────────────────────────────
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=5, padding=2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size,
                      padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class DoubleJPEGPaperCNN(nn.Module):
  
    def __init__(self, num_classes=2, qtable_dim=64,
                 c1=32, c2=64, c3=128, c4=128,
                 fc1_dim=256, fc2_dim=128):
        super().__init__()
        self.conv1 = ConvBNReLU(1,  c1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = ConvBNReLU(c1, c2)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = ConvBNReLU(c2, c3)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv4 = ConvBNReLU(c3, c4)
        self.fc1   = nn.LazyLinear(fc1_dim)
        self.fc2   = nn.Linear(fc1_dim + qtable_dim, fc2_dim)
        self.fc3   = nn.Linear(fc2_dim + qtable_dim, num_classes)

    def forward(self, hist, qtable):
        x = self.pool1(self.conv1(hist))
        x = self.pool2(self.conv2(x))
        x = self.pool3(self.conv3(x))
        x = self.conv4(x)
        x = torch.flatten(x, 1)
        x = torch.cat([x, qtable], dim=1)   # fusion ①
        x = F.relu(self.fc1(x))
        x = torch.cat([x, qtable], dim=1)   # fusion ②
        x = F.relu(self.fc2(x))
        x = torch.cat([x, qtable], dim=1)   # fusion ③
        return self.fc3(x)


# ── Feature Extraction ────────────────────────────────────────
def extract_features(path):
    try:
        jpg = jio.read(path)
        dct = jpg.coef_arrays[0][:256, :256]
        q   = jpg.quant_tables[0]
        q   = q.reshape(8, 8) if q.ndim == 1 else q

        if dct.shape != (256, 256):
            return None

        b    = 100
        bins = np.arange(-b, b + 2) - 0.5
        H    = np.zeros((64, 2 * b + 1), dtype=np.float32)
        Q    = np.zeros((64, 2 * b + 1), dtype=np.float32)

        for idx, (u, v) in enumerate(
                (u, v) for u in range(8) for v in range(8)):
            c         = np.clip(dct[u::8, v::8].flatten(), -b, b)
            H[idx], _ = np.histogram(c, bins=bins)
            Q[idx]    = float(q[u, v])   # scalar broadcast across 201 cols

        X  = np.stack([H, Q], axis=-1)   # [64, 201, 2]
        X /= (X.max() + 1e-6)
        return X
    except:
        return None


# ── Dataset ───────────────────────────────────────────────────
def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    hists, qtables, labels = zip(*batch)
    return torch.stack(hists), torch.stack(qtables), torch.tensor(labels)

class JPEGDataset(Dataset):
    def __init__(self, root_dir):
        self.file_list = []
        for label in ['single', 'double']:
            path = os.path.join(root_dir, label)
            if os.path.exists(path):
                for f in os.listdir(path):
                    if f.lower().endswith('.jpg'):
                        self.file_list.append(
                            (os.path.join(path, f),
                             0 if label == 'single' else 1))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label = self.file_list[idx]
        X = extract_features(path)
        if X is None:
            return None
        X      = torch.tensor(X).permute(2, 0, 1)  # [2, 64, 201]
        hist   = X[0].unsqueeze(0)                  # [1, 64, 201]
        qtable = X[1, :, 0]                         # [64]  (all cols identical)
        return hist, qtable, label


# ── Splits ────────────────────────────────────────────────────
full_ds  = JPEGDataset(EXTRACT_DIR)
total    = len(full_ds)
n_train  = int(0.30 * total)
n_val    = int(0.10 * total)
n_test   = int(0.10 * total)
n_unused = total - n_train - n_val - n_test

train_ds, val_ds, test_ds, _ = random_split(
    full_ds, [n_train, n_val, n_test, n_unused],
    generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          collate_fn=safe_collate, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                          collate_fn=safe_collate, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                          collate_fn=safe_collate, num_workers=2)

print(f"Train={len(train_ds)} | Val={len(val_ds)} | Test={len(test_ds)} | Unused={n_unused}")


# ── Evaluate ──────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    tp = tn = fp = fn = 0
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            inputs, qtables, labels = batch
            inputs  = inputs.to(DEVICE)
            qtables = qtables.to(DEVICE)
            labels  = labels.to(DEVICE)
            preds   = torch.argmax(model(inputs, qtables), dim=1)
            tp += ((preds == 1) & (labels == 1)).sum().item()
            tn += ((preds == 0) & (labels == 0)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
    acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    tpr = tp / (tp + fn + 1e-7)
    tnr = tn / (tn + fp + 1e-7)
    return acc, tpr, tnr


# ── Training ──────────────────────────────────────────────────
model     = DoubleJPEGPaperCNN().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))
criterion = nn.CrossEntropyLoss()
best_val_acc = 0.0
SAVE_PATH    = '/kaggle/working/best_model.pth'

# Initialise LazyLinear before training
with torch.no_grad():
    model(torch.zeros(1, 1, 64, 201).to(DEVICE),
          torch.zeros(1, 64).to(DEVICE))

print("Training Start (40 Epochs)...")
for epoch in range(40):

    if epoch == 30:
        for pg in optimizer.param_groups:
            pg['lr'] = 0.0005
        print("LR dropped to 0.0005")

    model.train()
    tp = tn = fp = fn = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        if batch is None:
            continue
        inputs, qtables, labels = batch
        inputs  = inputs.to(DEVICE)
        qtables = qtables.to(DEVICE)
        labels  = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(inputs, qtables)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    tpr = tp / (tp + fn + 1e-7)
    tnr = tn / (tn + fp + 1e-7)
    print(f"Epoch {epoch+1} | Train Acc={acc*100:.2f}% TPR={tpr*100:.2f}% TNR={tnr*100:.2f}%")

    va_acc, va_tpr, va_tnr = evaluate(model, val_loader)
    print(f"         | Val   Acc={va_acc*100:.2f}% TPR={va_tpr*100:.2f}% TNR={va_tnr*100:.2f}%")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"         | Best model saved (val={va_acc*100:.2f}%)")


# ── Final Test ────────────────────────────────────────────────
print("\nLoading best model for final test...")
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
te_acc, te_tpr, te_tnr = evaluate(model, test_loader)
print(f"\nTest | Acc={te_acc*100:.2f}% | TPR={te_tpr*100:.2f}% | TNR={te_tnr*100:.2f}%")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 MB 24.1 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for jpegio: filename=jpegio-0.2.8-cp312-cp312-linux_x86_64.whl size=1277276 sha256=87ffb62383b98d73f0005c6b38f6068254442e2b7e83f81cf2526872dadbe6a8
  Stored in directory: /root/.cache/pip/wheels/4e/5f/d9/3898c36bd7e9166a13a514d61f7ddd7310c192bfb443e758f8
Successfully built jpegio
Using Device: cuda
Streaming extraction started...


Processing: 610239it [03:51, 2639.84it/s]


Extraction Done: Single=40000, Double=40000
Train=24000 | Val=8000 | Test=8000 | Unused=40000
Training Start (40 Epochs)...


Epoch 1: 100%|██████████| 375/375 [01:09<00:00,  5.39it/s]

Epoch 1 | Train Acc=73.46% TPR=65.32% TNR=81.56%


...

Epoch 39 | Train Acc=93.11% TPR=88.69% TNR=97.51%


         | Val   Acc=85.55% TPR=88.31% TNR=82.82%


Epoch 40: 100%|██████████| 375/375 [01:08<00:00,  5.48it/s]

Epoch 40 | Train Acc=93.46% TPR=89.37% TNR=97.52%


         | Val   Acc=80.90% TPR=91.58% TNR=70.33%

Loading best model for final test...

Test | Acc=88.46% | TPR=83.30% | TNR=93.59%
